In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### VECTOR SEARCH with elasticsearch

In [11]:
es = Elasticsearch("http://localhost:9200")

In [ ]:

query = "Can I still join the course after the start date?"
v_query = model.encode(query).tolist() #this has to convert to a list before feeding the query into ES
v_query

[0.021390412002801895,
 -0.07397996634244919,
 0.0014206855557858944,
 0.021381661295890808,
 0.024511314928531647,
 0.03155827522277832,
 -0.11083970218896866,
 -0.10501749813556671,
 -0.06182589381933212,
 -0.006423118524253368,
 0.003723990637809038,
 0.09063936024904251,
 -0.009499410167336464,
 0.06539768725633621,
 0.011094655841588974,
 -0.02100973390042782,
 -0.033512551337480545,
 -0.043167732656002045,
 0.009963487274944782,
 0.014196986332535744,
 -0.06404148787260056,
 -0.007041821256279945,
 -0.07911880314350128,
 0.058003075420856476,
 0.001302120741456747,
 0.004197335336357355,
 0.057097915560007095,
 0.0639447569847107,
 0.02499029040336609,
 -0.03958766907453537,
 -0.037950679659843445,
 0.027039455249905586,
 0.017942320555448532,
 0.017227215692400932,
 0.03433111310005188,
 0.009290552698075771,
 0.05860547348856926,
 -0.049778956919908524,
 -0.005053660832345486,
 0.04343285411596298,
 -0.01566229574382305,
 -0.02975345030426979,
 -0.005133253522217274,
 0.0513414

In [29]:
filter_dict = {"course": "llm-zoomcamp"}

response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "knn": {
            "field": "vector"
            ,"query_vector": v_query
            ,"k": 5
            ,"num_candidates": 50
            ,"filter": {"term": filter_dict}
        }
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_score"])
    print(hit["_source"]["Q"], hit["_source"]["A"])
    print("---")


0.8268156
I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
---
0.75545925
When will the course be offered next? Summer 2027.
---
0.7231749
Certificate: Can I follow the course in a self-paced mode and get a certificate? No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.
---
0.71913683
Can I run the course locally instead of Codespaces? Yes. Codespaces is just the easiest way for everyone to start with the same environment.

You can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.

If you run l

### VECTOR SEARCH AGAINST DATASET USING argsort

In [3]:
documents = load_faq_data()

texts = [doc["question"].removeprefix('Course: ') + " " + doc["answer"] for doc in documents]

#chunk the dataset (texts) by batch. There are 27 batches given texts size is 13++ and each batch is 50. Encode by batch. 
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors), len(vectors[0])

X = np.array(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [4]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)
#This is matrix-vector multiplication. 
#Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

scores.shape

(1349,)

In [5]:
idx = np.argmax(scores) #returns the index of the highest value in the scores array
idx, scores[idx]


(np.int64(2), np.float32(0.801023))

In [6]:
documents[idx]
top5 = np.argsort(-scores)[:5]
for i in top5.tolist():
    print(scores[i], documents[i])

0.801023 {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
0.7579371 {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
0.7192132 {'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Qu

### BASIC VECTOR SEARCH: 1 TO 1

In [7]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

array([ 2.13904120e-02, -7.39799663e-02,  1.42068556e-03,  2.13816613e-02,
        2.45113149e-02,  3.15582752e-02, -1.10839702e-01, -1.05017498e-01,
       -6.18258938e-02, -6.42311852e-03,  3.72399064e-03,  9.06393602e-02,
       -9.49941017e-03,  6.53976873e-02,  1.10946558e-02, -2.10097339e-02,
       -3.35125513e-02, -4.31677327e-02,  9.96348727e-03,  1.41969863e-02,
       -6.40414879e-02, -7.04182126e-03, -7.91188031e-02,  5.80030754e-02,
        1.30212074e-03,  4.19733534e-03,  5.70979156e-02,  6.39447570e-02,
        2.49902904e-02, -3.95876691e-02, -3.79506797e-02,  2.70394552e-02,
        1.79423206e-02,  1.72272157e-02,  3.43311131e-02,  9.29055270e-03,
        5.86054735e-02, -4.97789569e-02, -5.05366083e-03,  4.34328541e-02,
       -1.56622957e-02, -2.97534503e-02, -5.13325352e-03,  5.13414629e-02,
        6.16060290e-03,  6.86980635e-02, -1.29505778e-02, -5.61938696e-02,
       -1.08265011e-02,  5.96683845e-02,  5.29939681e-02, -3.42755206e-02,
       -4.15274203e-02, -

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv

array([-4.81206626e-02, -1.29942089e-01, -4.86809993e-03,  1.37495315e-02,
       -8.51100218e-03,  6.97841346e-02, -7.55665451e-03,  4.09571715e-02,
       -1.02165632e-01, -3.45050693e-02,  2.85521429e-02, -5.81430867e-02,
       -1.85113307e-02, -3.17839049e-02,  5.29630929e-02,  3.96292843e-02,
       -9.84596927e-03, -6.45324662e-02,  7.05097690e-02, -6.39589354e-02,
       -3.55157033e-02, -1.75490268e-02, -3.58401537e-02,  5.50837722e-04,
        3.38445492e-02,  2.91762855e-02,  3.24278772e-02, -5.64803649e-03,
        8.42593089e-02,  5.90306558e-02,  2.08773166e-02, -5.44832181e-03,
        2.98403613e-02,  1.30820693e-02,  6.36077821e-02, -3.06394324e-02,
        6.28459901e-02, -1.69217791e-02, -4.65758629e-02, -5.13762161e-02,
       -3.81560996e-02, -7.01197386e-02, -9.55517516e-02,  5.01640290e-02,
        8.57914891e-03,  1.19207576e-02, -1.40597168e-02, -7.98975956e-03,
       -3.44238691e-02, -2.34269854e-02,  2.89466102e-02,  2.68904660e-02,
       -4.87408601e-02, -

In [9]:
# we compare the query against the document using dot product:
v1.dot(dv)

np.float32(0.32332397)

In [10]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730438)